In [0]:
%run ./_config

In [0]:
-- inspect table history and details
DESCRIBE HISTORY ${env}.silver.fct_sales;
DESCRIBE DETAIL ${env}.silver.fct_sales;

In [0]:
-- compact small files and co-locate by most filtered columns
OPTIMIZE ${env}.silver.fct_sales
ZORDER BY (sale_date, store_id);

In [0]:
-- clean up old files
VACUUM ${env}.silver.fct_sales DRY RUN;
VACUUM ${env}.silver.fct_sales RETAIN 168 HOURS;

In [ ]:
-- CDF audit, tracks what changed in fct_sales per pipeline run
-- writes to gold.daily_load_audit 

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.daily_load_audit (
        load_date DATE,
        change_type STRING,
        commit_version BIGINT,
        commit_ts TIMESTAMP,
        rows_affected BIGINT,
        dollars_affected DECIMAL(18,2),
        bottles_affected BIGINT,
        earliest_sale DATE,
        latest_sale DATE
    )
    TBLPROPERTIES ('quality' = 'gold')
""")

# get the last audited version to avoid reprocessing
last_audited = spark.sql(f"""
    SELECT COALESCE(MAX(commit_version), 0) AS v
    FROM {GOLD_SCHEMA}.daily_load_audit
""").collect()[0]["v"]

# read new changes since last audit
try:
    changes_df = spark.sql(f"""
        SELECT
            DATE(_commit_timestamp) AS load_date,
            _change_type AS change_type,
            _commit_version AS commit_version,
            MIN(_commit_timestamp) AS commit_ts,
            COUNT(*) AS rows_affected,
            SUM(sale_dollars) AS dollars_affected,
            SUM(bottles_sold) AS bottles_affected,
            MIN(sale_date) AS earliest_sale,
            MAX(sale_date) AS latest_sale
        FROM table_changes('{SILVER_SCHEMA}.fct_sales', {last_audited + 1})
        GROUP BY DATE(_commit_timestamp), _change_type, _commit_version
    """)

    new_batches = changes_df.count()
    if new_batches > 0:
        changes_df.write.mode("append").saveAsTable(f"{GOLD_SCHEMA}.daily_load_audit")
        print(f"CDF audit: {new_batches} change batches written since version {last_audited}")
        changes_df.orderBy("commit_version").display()
    else:
        print(f"CDF audit: no new changes since version {last_audited}")

except Exception as e:
    if "not found" in str(e).lower() or "version" in str(e).lower():
        print(f"CDF audit: no changes available after version {last_audited} — {e}")
    else:
        raise